## Column Generation example

In [ ]:
using Pkg

Pkg.activate(".")
Pkg.instantiate()

In [ ]:
using HiGHS
using JuMP
using Graphs
using GraphPlot
using Plots
using LinearAlgebra
using Random

In [ ]:
include("pdp.jl")

## Problema aleatorios 

In [ ]:
function problema_aleatorio(altura, largura, m, r, num_caminhoes, L)
    # gerando as cidades aleatorias 
    cid = Array{Float64}(undef, m, 2)
    for i = 1:m 
        cid[i, 1] = largura*rand()
        cid[i, 2] = altura*rand()
    end

    # calculando a distancia entre todas as cidades 
    C = Array{Float64}(undef, m, m)
    for i = 1:m
        C[i, i] = 0
    end
    for t = 1:m-1
        for s = t+1:m
            a = cid[t, 1]
            b = cid[t, 2]
            c = cid[s, 1]
            d = cid[s, 2]
            C[t, s] = sqrt((a-c)^2 + (b-d)^2)
            C[s,t]=C[t, s]
        end
    end

    # gerando uma quantidade r de tarefas 
    task = Array{Int64}(undef, r, 2)
    for i = 1:r
        a = rand(1:m)
        b = rand(1:m)
        while b == a 
            b = rand(1:m)
        end
        task[i, 1] = a
        task[i, 2] = b
    end

    # gerando uma quantidade r de janelas de tempo para realizacao de cada tarefa 
    W = Array{Float64}(undef, r, 2)
    for i = 1:r
        origem = task[i, 1]
        destino = task[i, 2]
        distancia = C[origem, destino]  # distância entre as cidades da tarefa
        c = distancia + (L - distancia) * rand()
        d = c + (L - c) * rand()     
        W[i, 1] = c
        W[i, 2] = d
    end
    return C, task, W
end

## Grafo

In [ ]:
function grafo(C, task, r)
    # Identificar apenas as cidades usadas nas tarefas
    usados = unique(vcat(task[:, 1], task[:, 2]))
    mapa_cidade = Dict(cidade => i for (i, cidade) in enumerate(usados))

    # Criar grafo apenas com as cidades usadas
    g = SimpleDiGraph(length(usados))

    # Criar matriz truncada inicializada com zeros
    C_truncada = zeros(Float64, length(usados), length(usados))

    # Preencher matriz apenas com distâncias das tarefas
    for i = 1:r
        origem_real = task[i, 1]
        destino_real = task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        C_truncada[origem, destino] = trunc(C[origem_real, destino_real])
    end

    # Criar lista de arestas com pesos
    weights = Dict()
    for i in 1:size(task, 1)
        origem_real, destino_real = task[i, 1], task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        if C_truncada[origem, destino] > 0
            add_edge!(g, origem, destino)
            weights[(origem, destino)] = C_truncada[origem, destino]
        end
    end

    # Obter rótulos das arestas
    graph_edges = collect(Graphs.edges(g))
    edge_labels = [weights[(u.src, u.dst)] for u in graph_edges]
    
    # Plotar o grafo (números originais das cidades no rótulo)
    #gplot(g, nodelabel=usados, edgelabel=edge_labels)
    display(gplot(g, nodelabel=usados, edgelabel=edge_labels))
    return g, weights
end

## Dados de entrada

In [ ]:
# Representacao ficticia do Parana 
altura = 50
largura = 70

# m vai ser a quantidade de cidades dentro do Parana 
m = 10

# r vai ser a quantidade de tarefas que vou ter 
r = 10

# numero de caminhoes
num_caminhoes = 2

# Limite de tempo
L = 200

# Semente 87213 2 3 
numero_primo = 	87213
teste = 2
semente = numero_primo + teste
Random.seed!(semente)

In [ ]:
C, task, W = problema_aleatorio(altura, largura, m, r, num_caminhoes, L)

In [ ]:
A, solucao, g = A_final(C, task, W, num_caminhoes, r)

In [ ]:
println("É inteira? $(all(isinteger.(solucao)))")

In [ ]:
rota_da_solucao(solucao, A, g)